# 53. PDF 조건부 열접촉 후보 예측 감사·가스17개 설명
52번 예측에서 2×2를 재집계하고 라벨 일치·기존 탐지 유지·중복 없는 평가를 확인한다. 가스17개 조건부 정상 대비 편차를 저장한다. 센서 기준의 연관 패턴이며 가스 원인/세부유형 정답으로 해석하지 않는다.


In [ ]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
nb=json.loads((root/'notebooks/52_rg3_pdf_thermal_seed_ensemble.ipynb').read_text(encoding='utf-8'))
exec(''.join(nb['cells'][1]['source']).split('memo={}')[0])
pred=pd.read_csv(tables/'rg3_pdf_thermal_seed_predictions.csv',parse_dates=['TimeStamp'])
verification=[]
for key,g in pred.groupby(['scheme','model','budget','combine'],sort=False):
 assert not g.TimeStamp.duplicated().any()
 assert ((g.anchor_prediction==0)|(g.prediction==1)).all() if key[3] in ['OR_anchor','fixed'] and key[1]!='baseline_RF' else True
 actual=d.set_index('TimeStamp').loc[g.TimeStamp,'PassOrFail'].to_numpy(int)
 assert np.array_equal(actual,g.actual.to_numpy())
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 verification.append(dict(scheme=key[0],model=key[1],budget=key[2],combine=key[3],TP=tp,FP=fp,FN=fn,TN=tn,F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0))
v=pd.DataFrame(verification)
v.to_csv(tables/'rg3_pdf_thermal_verification.csv',index=False,encoding='utf-8-sig')
audit=[]
for b in blocks:
 q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b))
 assert not set(d.TimeStamp.iloc[q])&set(d.TimeStamp.iloc[ref])
 z=local_residual(ref,q,'surface_thermal_contact')
 for j,i in enumerate(q):
  record=dict(TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]))
  for k,f in enumerate(F['surface_thermal_contact']):
   record[f+'__conditional_z']=z[j,k]
  audit.append(record)
a=pd.DataFrame(audit)
candidate=pred[(pred.scheme=='leave_one_block_out')&(pred.model=='surface_thermal_contact__tree_linear_mean')&(pred.budget==.025)&(pred['combine']=='OR_anchor')]
a=a.merge(candidate[['TimeStamp','prediction','anchor_prediction','score','threshold']],on='TimeStamp',validate='one_to_one')
a.to_csv(tables/'rg3_pdf_thermal_all_shot_diagnostics.csv',index=False,encoding='utf-8-sig')
a[a.actual.eq(1)].to_csv(tables/'rg3_pdf_thermal_all17_diagnostics.csv',index=False,encoding='utf-8-sig')
print('New candidate all gas cases',a[a.actual.eq(1)][['TimeStamp','anchor_prediction','prediction','score','threshold']].to_string(index=False))
print('Newly detected gas conditional deviations',a[(a.actual==1)&(a.prediction==1)&(a.anchor_prediction==0)].to_string(index=False))
print('Normal conditional deviations: medians and 10/90 percentiles')
print(a[a.actual==0].filter(like='__conditional_z').quantile([.1,.5,.9]).to_string())
print('Verified primary mean candidate',v[(v.model=='surface_thermal_contact__tree_linear_mean')&(v.budget==.025)&(v['combine']=='OR_anchor')].to_string(index=False))
